# Real-World Data Project: Retail Business Analytics

## Objective
Perform an end-to-end analysis of a retail business dataset to understand sales performance, profitability, customer behavior, regional performance, and operational efficiency.

## Business Questions
1. Which product categories generate the most revenue and profit?
2. Which regions and sales channels perform best?
3. How do discounts affect profit margins?
4. What is the monthly sales and profit trend?
5. Is delivery time associated with customer ratings?
6. Which payment methods and categories are most common?

**Workflow:** Business Problem → Data Quality → Cleaning → KPI Analysis → Visualization → Business Recommendations


## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 2. Load the Retail Dataset

In [ ]:
df = pd.read_csv("retail_business_dataset.csv", parse_dates=["Order_Date"])

print("Dataset shape:", df.shape)
display(df.head())


## 3. Data Quality Assessment

In [ ]:
df.info()


In [ ]:
quality_report = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isna().sum(),
    "Missing %": (df.isna().mean()*100).round(2),
    "Unique Values": df.nunique()
})
display(quality_report)
print("Duplicate rows:", df.duplicated().sum())


## 4. Data Cleaning

In [ ]:
clean_df = df.copy()

# Remove exact duplicate rows
before = len(clean_df)
clean_df = clean_df.drop_duplicates().reset_index(drop=True)

# Fill numerical missing values with median
for column in clean_df.select_dtypes(include=np.number).columns:
    clean_df[column] = clean_df[column].fillna(clean_df[column].median())

# Fill categorical missing values with mode
for column in clean_df.select_dtypes(exclude=np.number).columns:
    if column != "Order_Date":
        mode = clean_df[column].mode()
        if not mode.empty:
            clean_df[column] = clean_df[column].fillna(mode.iloc[0])

print("Duplicate rows removed:", before - len(clean_df))
print("Remaining missing values:", clean_df.isna().sum().sum())
print("Cleaned shape:", clean_df.shape)


## 5. Business KPIs

In [ ]:
kpis = pd.Series({
    "Total Revenue": clean_df["Net_Sales"].sum(),
    "Total Profit": clean_df["Profit"].sum(),
    "Total Units Sold": clean_df["Units_Sold"].sum(),
    "Average Order Value": clean_df["Net_Sales"].mean(),
    "Average Profit Margin (%)": clean_df["Profit_Margin"].mean(),
    "Average Customer Rating": clean_df["Customer_Rating"].mean(),
    "Average Delivery Days": clean_df["Delivery_Days"].mean(),
    "Number of Orders": clean_df["Order_ID"].nunique()
})

display(kpis.to_frame("Value"))


## 6. Product Category Performance

In [ ]:
category_summary = (
    clean_df.groupby("Product_Category")
    .agg(
        Revenue=("Net_Sales","sum"),
        Profit=("Profit","sum"),
        Units=("Units_Sold","sum"),
        Avg_Rating=("Customer_Rating","mean")
    )
    .sort_values("Revenue", ascending=False)
)

display(category_summary.round(2))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

category_summary["Revenue"].plot(kind="bar", ax=axes[0])
axes[0].set_title("Revenue by Product Category")
axes[0].set_xlabel("Product Category")
axes[0].set_ylabel("Revenue")
axes[0].tick_params(axis="x", rotation=25)

category_summary["Profit"].plot(kind="bar", ax=axes[1])
axes[1].set_title("Profit by Product Category")
axes[1].set_xlabel("Product Category")
axes[1].set_ylabel("Profit")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()


## 7. Regional and Channel Analysis

In [ ]:
region_summary = clean_df.groupby("Region").agg(
    Revenue=("Net_Sales","sum"),
    Profit=("Profit","sum"),
    Orders=("Order_ID","nunique")
).sort_values("Revenue", ascending=False)

channel_summary = clean_df.groupby("Sales_Channel").agg(
    Revenue=("Net_Sales","sum"),
    Profit=("Profit","sum"),
    Orders=("Order_ID","nunique")
).sort_values("Revenue", ascending=False)

display(region_summary.round(2))
display(channel_summary.round(2))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))
region_summary["Revenue"].plot(kind="bar", ax=axes[0])
axes[0].set_title("Revenue by Region")
axes[0].set_ylabel("Revenue")

channel_summary["Revenue"].plot(kind="bar", ax=axes[1])
axes[1].set_title("Revenue by Sales Channel")
axes[1].set_ylabel("Revenue")

plt.tight_layout()
plt.show()


## 8. Monthly Sales and Profit Trend

In [ ]:
monthly = (
    clean_df.set_index("Order_Date")
    .resample("ME")[["Net_Sales","Profit"]]
    .sum()
)

display(monthly.head())


In [ ]:
monthly.plot(figsize=(12,5), marker="o")
plt.title("Monthly Revenue and Profit Trend")
plt.xlabel("Month")
plt.ylabel("Amount")
plt.tight_layout()
plt.show()


## 9. Discount and Profitability Analysis

In [ ]:
discount_summary = clean_df.groupby("Discount").agg(
    Revenue=("Net_Sales","sum"),
    Profit=("Profit","sum"),
    Avg_Margin=("Profit_Margin","mean"),
    Orders=("Order_ID","nunique")
)

display(discount_summary.round(2))


In [ ]:
plt.figure(figsize=(9,5))
sns.boxplot(data=clean_df, x="Discount", y="Profit_Margin")
plt.title("Profit Margin Distribution by Discount")
plt.xlabel("Discount")
plt.ylabel("Profit Margin (%)")
plt.show()


## 10. Customer Experience and Operations

In [ ]:
plt.figure(figsize=(9,5))
sns.scatterplot(
    data=clean_df,
    x="Delivery_Days",
    y="Customer_Rating",
    hue="Sales_Channel",
    alpha=.7
)
plt.title("Delivery Time vs Customer Rating")
plt.xlabel("Delivery Days")
plt.ylabel("Customer Rating")
plt.show()


In [ ]:
plt.figure(figsize=(9,5))
sns.countplot(
    data=clean_df,
    x="Payment_Method",
    order=clean_df["Payment_Method"].value_counts().index
)
plt.title("Orders by Payment Method")
plt.xlabel("Payment Method")
plt.ylabel("Number of Orders")
plt.xticks(rotation=20)
plt.show()


## 11. Correlation Analysis

In [ ]:
numeric_columns = [
    "Units_Sold", "Unit_Price", "Discount", "Customer_Rating",
    "Delivery_Days", "Inventory_Available", "Net_Sales",
    "Profit", "Profit_Margin"
]

plt.figure(figsize=(10,8))
sns.heatmap(
    clean_df[numeric_columns].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Retail Business Correlation Matrix")
plt.tight_layout()
plt.show()


## 12. Business Recommendations

In [ ]:
best_category = category_summary.index[0]
best_region = region_summary.index[0]
best_channel = channel_summary.index[0]
best_margin_discount = discount_summary["Avg_Margin"].idxmax()

recommendations = [
    f"Prioritize inventory and marketing for the highest-revenue category: {best_category}.",
    f"Study successful practices in the best-performing region: {best_region}.",
    f"Review customer acquisition and operational strategy for the leading channel: {best_channel}.",
    f"Use discount levels carefully and monitor profit margin before expanding promotions.",
    "Track delivery performance because customer experience can influence repeat purchases.",
    "Use category-level profit, not revenue alone, for business decisions."
]

for i, recommendation in enumerate(recommendations, 1):
    print(f"{i}. {recommendation}")


## Conclusion

This project applies data-science techniques to a retail business context.

The analysis covers data cleaning, business KPIs, category performance, regional and channel comparisons, time-series trends, discount profitability, customer experience, correlation analysis, and actionable recommendations.

**Important:** The dataset is a realistic practice dataset created for demonstration. It is not confidential company data and should not be presented as actual business records.
